# Optimizing a Walker with Genetic Algorithms

The Strider is parameterized by seven link lengths. We score candidates
on a cheap **kinematic** fitness (`StrideFitness`) with a small GA,
then look at before/after foot loci.

**What you'll learn:**
- Parameterizing a factory walker for optimization
- Running `genetic_algorithm_optimization` on a kinematic objective
- Reading an `Ensemble` result (`top`, `filter_by_score`)
- When to chain a GA with `chain_walking_optimizers` for local polish


In [ ]:
import warnings

import matplotlib.pyplot as plt

import leggedsnake as ls
from pylinkage import extract_trajectory
from pylinkage.optimization import generate_bounds

warnings.filterwarnings('ignore', category=DeprecationWarning)


## 1. The starting Strider

`Walker.from_strider(...)` takes seven lengths. Most of pylinkage's
optimizers talk to a linkage via its `.joints` list (dimensions per
joint). Walker exposes `.joints` for exactly this reason.


In [ ]:
START = dict(crank=1.0, triangle=2.0, femur=1.8, rocker_l=2.6,
             rocker_s=1.4, tibia=2.5, foot=1.8)

def make_strider():
    return ls.Walker.from_strider(**START)

prototype = make_strider()
print(f"Strider start: DOF={prototype.dof}, feet={prototype.get_feet()}")
edge_names = list(prototype.dimensions.edge_distances.keys())
dim_values = list(prototype.dimensions.edge_distances.values())
print(f"{len(dim_values)} free edge distances to optimize")


## 2. A kinematic fitness on the Strider foot locus

`StrideFitness` returns the horizontal stride length (the portion of
the foot locus that stays below a given height — i.e., the stance
phase). We adapt it to the GA's `(eval_func(dna))` contract with
`as_ga_fitness`.


In [ ]:
fit = ls.StrideFitness(step_height=0.5, stride_height=0.2, foot_index=-1)
r = fit(prototype.topology, prototype.dimensions, ls.WorldConfig())
print(f"Starting stride score: {r.score:.3f}  (valid={r.valid})")

# Adapt to the optimizer contract: (linkage, dims, pos) -> float
eval_func = ls.as_eval_func(fit, walker_factory=make_strider)


## 3. Search — `genetic_algorithm_optimization`

Small population and few iterations keep the demo fast. On a real
problem use `max_pop≈40, iters≈80+` and multiprocessing via
`processes=...`.


In [ ]:
bounds = generate_bounds(dim_values, min_ratio=2.0, max_factor=2.0)

ensemble = ls.genetic_algorithm_optimization(
    eval_func=eval_func,
    linkage=prototype,
    center=dim_values,
    bounds=bounds,
    max_pop=10,
    iters=4,
    processes=1,
    verbose=False,
)

print(f"Ensemble size: {len(ensemble)}")
for i, agent in enumerate(ensemble.top(3)):
    print(f"  rank {i}: score={agent.score:.3f}")


## 4. Visualize before vs after

Plot the starting and best-after-GA foot loci side-by-side.


In [ ]:
def foot_xy(walker):
    mech = walker.to_mechanism()
    loci = list(walker.step(iterations=120))
    xs, ys = extract_trajectory(loci, len(mech.joints) - 1)
    return xs, ys

best_agent = ensemble[0]
# Build a fresh Strider, then swap in the optimized edge distances.
best_walker = make_strider()
best_walker.set_num_constraints(list(best_agent.dimensions))

fig, ax = plt.subplots(figsize=(7, 4))
xs, ys = foot_xy(prototype)
ax.plot(xs, ys, '--', color='#888', label=f"start (stride={r.score:.2f})")
xs, ys = foot_xy(best_walker)
ax.plot(xs, ys, '-', color='#c44e52', label=f"best GA (stride={best_agent.score:.2f})")
ax.set_aspect('equal'); ax.grid(True); ax.legend()
ax.set_title("Strider foot locus — GA before/after")
plt.show()


## 5. Ensemble inspection

`Ensemble` wraps the Pareto-ordered population with helpers: `top(n)`,
`filter_by_score(min_score=...)`. Handy for checkpointing or reseeding
a follow-up run.


In [ ]:
survivors = ensemble.filter_by_score(min_val=r.score)
print(f"{len(survivors)} / {len(ensemble)} candidates beat the starting score")


## Summary

- `genetic_algorithm_optimization` takes a *Walker* via `linkage=`
  and a `([lo...], [hi...])` bounds tuple, and returns a pylinkage
  `Ensemble`.
- `StrideFitness` + `as_ga_fitness` is the cheap kinematic pairing —
  budget this one in the inner loop, physics fitness in the outer.
- **Chaining**: `chain_walking_optimizers(fitness, walker, stages=[...])`
  feeds each stage the previous best — the go-to pattern for *global
  search → local polish* (differential evolution → Nelder-Mead).

Next (notebook 04): multi-objective NSGA-II over distance and
efficiency, plus gait analysis on the Pareto winners.
